In [15]:
from google.colab import files

archivos_subidos = files.upload()
print(list(archivos_subidos.keys()))

Saving cloud_provider_challenge_dataset_v1.zip to cloud_provider_challenge_dataset_v1.zip
['cloud_provider_challenge_dataset_v1.zip']


In [16]:
from pathlib import Path
import zipfile

zip_path = Path("/content/cloud_provider_challenge_dataset_v1.zip")
carpeta_datos = Path("/content/cloud_provider")
carpeta_datos.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(carpeta_datos)

print("CSV encontrados:")
for archivo in sorted(carpeta_datos.rglob("*.csv")):
    print(archivo.relative_to(carpeta_datos))

CSV encontrados:
datalake/landing/billing_monthly.csv
datalake/landing/customers_orgs.csv
datalake/landing/marketing_touches.csv
datalake/landing/nps_surveys.csv
datalake/landing/resources.csv
datalake/landing/support_tickets.csv
datalake/landing/users.csv


Voy a revisar las columnas y un par de filas de los tres CSV elegidos, para definir los esquemas de Spark con los nombres reales.

In [17]:
from pathlib import Path
import csv

landing = Path("/content/cloud_provider/datalake/landing")
nombres = [
    "customers_orgs.csv",
    "users.csv",
    "billing_monthly.csv",
]

for nombre in nombres:
    ruta = landing / nombre
    print(f"\n--- {nombre} ---")
    with ruta.open("r", encoding="utf-8", newline="") as archivo:
        lector = csv.reader(archivo)
        print("Columnas:", next(lector))
        for _, fila in zip(range(2), lector):
            print(fila)


--- customers_orgs.csv ---
Columnas: ['org_id', 'org_name', 'industry', 'hq_region', 'plan_tier', 'is_enterprise', 'signup_date', 'sales_rep', 'lifecycle_stage', 'marketing_source', 'nps_score']
['org_xaji0y6d', 'Nimbus Labs 0', 'Education', 'sa-east', 'standard', 'True', '2025-05-26', 'rep_c', 'churned', 'partner', '']
['org_pbhsahxt', 'Nova Tech 1', 'Education', 'us-east', 'standard', 'False', '2025-06-16', 'rep_d', 'lead', 'event', '-3.0']

--- users.csv ---
Columnas: ['user_id', 'org_id', 'email', 'role', 'active', 'created_at', 'last_login']
['user_9tze5r5u', 'org_3t60rjiw', 'user_9tze5r5u@example.com', 'devops', 'True', '2025-07-10', '']
['user_qpgb7r3o', 'org_afeyuhz1', 'user_qpgb7r3o@example.com', 'developer', 'True', '2025-07-15', '2025-08-10']

--- billing_monthly.csv ---
Columnas: ['invoice_id', 'org_id', 'month', 'subtotal', 'credits', 'taxes', 'currency', 'exchange_rate_to_usd']
['inv_nym31sk0', 'org_xaji0y6d', '2025-06-01', '1133.34', '', '238.0', 'USD', '0.97684']
['inv

Instalacion de pyspark

In [18]:
!pip -q install pyspark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CloudProviderMVP")
    .getOrCreate()
)

print("Spark iniciado. Versión:", spark.version)

Spark iniciado. Versión: 4.0.5


Revisamos los archivos

In [19]:
from pathlib import Path
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    DateType, DoubleType, DecimalType
)

landing = Path("/content/cloud_provider/datalake/landing")

esquemas = {
    "customers_orgs.csv": StructType([
        StructField("org_id", StringType(), True),
        StructField("org_name", StringType(), True),
        StructField("industry", StringType(), True),
        StructField("hq_region", StringType(), True),
        StructField("plan_tier", StringType(), True),
        StructField("is_enterprise", BooleanType(), True),
        StructField("signup_date", DateType(), True),
        StructField("sales_rep", StringType(), True),
        StructField("lifecycle_stage", StringType(), True),
        StructField("marketing_source", StringType(), True),
        StructField("nps_score", DoubleType(), True),
    ]),

    "users.csv": StructType([
        StructField("user_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("email", StringType(), True),
        StructField("role", StringType(), True),
        StructField("active", BooleanType(), True),
        StructField("created_at", DateType(), True),
        StructField("last_login", DateType(), True),
    ]),

    "billing_monthly.csv": StructType([
        StructField("invoice_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("month", DateType(), True),
        StructField("subtotal", DecimalType(18, 2), True),
        StructField("credits", DecimalType(18, 2), True),
        StructField("taxes", DecimalType(18, 2), True),
        StructField("currency", StringType(), True),
        StructField("exchange_rate_to_usd", DecimalType(12, 5), True),
    ]),
}

tablas = {}

for nombre, esquema in esquemas.items():
    df = (
        spark.read
        .option("header", True)
        .option("dateFormat", "yyyy-MM-dd")
        .schema(esquema)
        .csv(str(landing / nombre))
    )

    tablas[nombre.removesuffix(".csv")] = df

    print(f"\n--- {nombre}: {df.count()} filas ---")
    df.printSchema()


--- customers_orgs.csv: 80 filas ---
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)


--- users.csv: 800 filas ---
root
 |-- user_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- role: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_login: date (nullable = true)


--- billing_monthly.csv: 240 filas ---
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: decimal(18,2

Hacemos la ingesta en BRONZE como parquet, agregando ingest_ts y source_file. Uso ingest_date como partición,y evito particionar por org_id, que generaría muchas particiones pequeñas.

In [20]:
from pathlib import Path
from pyspark.sql.functions import current_timestamp, to_date, lit

ruta_bronze = Path("/content/cloud_provider/datalake/bronze")
ruta_bronze.mkdir(parents=True, exist_ok=True)

for nombre, df in tablas.items():
    archivo_origen = f"{nombre}.csv"
    destino = ruta_bronze / nombre

    df_bronze = (
        df
        .withColumn("ingest_ts", current_timestamp())
        .withColumn("ingest_date", to_date("ingest_ts"))
        .withColumn("source_file", lit(archivo_origen))
    )

    (
        df_bronze.write
        .mode("overwrite")
        .partitionBy("ingest_date")
        .parquet(str(destino))
    )

    verificacion = spark.read.parquet(str(destino))
    print(
        f"{nombre}: {verificacion.count()} filas escritas en "
        f"{destino}"
    )

customers_orgs: 80 filas escritas en /content/cloud_provider/datalake/bronze/customers_orgs
users: 800 filas escritas en /content/cloud_provider/datalake/bronze/users
billing_monthly: 240 filas escritas en /content/cloud_provider/datalake/bronze/billing_monthly


**overwrite reemplaza la salida de cada tabla cuando volvés a ejecutar esta carga completa, en lugar de ir agregando las mismas filas.

Hago una revision rapida para ver si tenemos los campos correctamente aprovisionados.

In [21]:
for nombre in tablas:
    ruta = ruta_bronze / nombre
    df = spark.read.parquet(str(ruta))

    carpetas_particion = [
        p.name for p in ruta.iterdir()
        if p.is_dir() and p.name.startswith("ingest_date=")
    ]

    archivos_parquet = list(ruta.rglob("*.parquet"))

    print(f"\n--- {nombre} ---")
    print("Filas:", df.count())
    print("Columnas técnicas presentes:",
          {"ingest_ts", "ingest_date", "source_file"}.issubset(df.columns))
    print("Particiones:", carpetas_particion)
    print("Archivos Parquet:", len(archivos_parquet))

    df.select("ingest_ts", "ingest_date", "source_file").show(1, truncate=False)


--- customers_orgs ---
Filas: 80
Columnas técnicas presentes: True
Particiones: ['ingest_date=2026-11-18']
Archivos Parquet: 1
+--------------------------+-----------+------------------+
|ingest_ts                 |ingest_date|source_file       |
+--------------------------+-----------+------------------+
|2026-11-18 21:16:36.386772|2026-11-18 |customers_orgs.csv|
+--------------------------+-----------+------------------+
only showing top 1 row

--- users ---
Filas: 800
Columnas técnicas presentes: True
Particiones: ['ingest_date=2026-11-18']
Archivos Parquet: 1
+--------------------------+-----------+-----------+
|ingest_ts                 |ingest_date|source_file|
+--------------------------+-----------+-----------+
|2026-11-18 21:16:39.727397|2026-11-18 |users.csv  |
+--------------------------+-----------+-----------+
only showing top 1 row

--- billing_monthly ---
Filas: 240
Columnas técnicas presentes: True
Particiones: ['ingest_date=2026-11-18']
Archivos Parquet: 1
+----------

Busco JSONL para preparar la ingesta streaming.

In [23]:
from pathlib import Path

carpeta_datos = Path("/content/cloud_provider")
archivos_jsonl = sorted(carpeta_datos.rglob("*.jsonl"))

print("Cantidad de archivos JSONL:", len(archivos_jsonl))
print("Primeros archivos:")
for archivo in archivos_jsonl[:5]:
    print(archivo.relative_to(carpeta_datos))

Cantidad de archivos JSONL: 120
Primeros archivos:
datalake/landing/usage_events_stream/events_part_0000.jsonl
datalake/landing/usage_events_stream/events_part_0001.jsonl
datalake/landing/usage_events_stream/events_part_0002.jsonl
datalake/landing/usage_events_stream/events_part_0003.jsonl
datalake/landing/usage_events_stream/events_part_0004.jsonl


Tenemos un segundo esquema que agrega informacion, hago la validacion de estos registros y genero un nuevo registro de prueba para consolidas ambas versiones.

In [24]:
import json

muestras_version = {}
muestra_genai = None

for archivo in archivos_jsonl:
    with archivo.open("r", encoding="utf-8") as f:
        for linea in f:
            evento = json.loads(linea)
            version = str(evento.get("schema_version"))

            if version not in muestras_version:
                muestras_version[version] = evento

            if (
                muestra_genai is None
                and evento.get("genai_tokens") not in (None, "")
            ):
                muestra_genai = evento

            if len(muestras_version) >= 2 and muestra_genai is not None:
                break

    if len(muestras_version) >= 2 and muestra_genai is not None:
        break

for version, evento in muestras_version.items():
    print(f"\n--- schema_version {version} ---")
    print(json.dumps(evento, indent=2, ensure_ascii=False))

print("\n--- Ejemplo con genai_tokens ---")
print(json.dumps(muestra_genai, indent=2, ensure_ascii=False))


--- schema_version 2 ---
{
  "event_id": "evt_nwlh5boz2guf",
  "timestamp": "2025-08-17T01:55:00Z",
  "org_id": "org_cvs4f8cg",
  "resource_id": "res_i8kcmm7d",
  "service": "networking",
  "region": "sa-east",
  "metric": "requests",
  "value": null,
  "unit": "count",
  "cost_usd_increment": 0.9561,
  "schema_version": 2,
  "carbon_kg": 0
}

--- schema_version 1 ---
{
  "event_id": "evt_xozb632zko0g",
  "timestamp": "2025-07-16T11:20:00Z",
  "org_id": "org_rixa11dp",
  "resource_id": "res_nmk7yyz6",
  "service": "database",
  "region": "ap-northeast",
  "metric": "requests",
  "value": null,
  "unit": "count",
  "cost_usd_increment": 5.6521,
  "schema_version": 1
}

--- Ejemplo con genai_tokens ---
{
  "event_id": "evt_v64of7mqdgpy",
  "timestamp": "2025-08-08T15:19:00Z",
  "org_id": "org_rixa11dp",
  "resource_id": "res_a09w1gxh",
  "service": "genai",
  "region": "us-east",
  "metric": "requests",
  "value": 123.0,
  "unit": "count",
  "cost_usd_increment": 13.801,
  "schema_versi

El segundo esquema agrega el campo carbon_kg y para algunos registros genai_tokens por lo que se agregaran como opcionales. value quedará como texto en Bronze para preservar sus valores numéricos y textuales; lo convierto después en Silver.

Realizo validacion adicional.

In [25]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DoubleType, IntegerType, LongType
)
from pyspark.sql.functions import col

spark.conf.set("spark.sql.session.timeZone", "UTC")

carpeta_eventos = (
    "/content/cloud_provider/datalake/landing/usage_events_stream"
)

esquema_eventos = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("org_id", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("metric", StringType(), True),
    StructField("value", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("schema_version", IntegerType(), True),
    StructField("carbon_kg", DoubleType(), True),
    StructField("genai_tokens", LongType(), True),
])

eventos_prueba = (
    spark.read
    .schema(esquema_eventos)
    .json(carpeta_eventos)
)

print("Total de eventos:", eventos_prueba.count())
print("Eventos por versión:")
eventos_prueba.groupBy("schema_version").count().orderBy("schema_version").show()

print(
    "Eventos sin timestamp:",
    eventos_prueba.filter(col("timestamp").isNull()).count()
)

eventos_prueba.select(
    "event_id", "timestamp", "schema_version",
    "value", "carbon_kg", "genai_tokens"
).show(5, truncate=False)

Total de eventos: 43200
Eventos por versión:
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+

Eventos sin timestamp: 0
+----------------+-------------------+--------------+-------+---------+------------+
|event_id        |timestamp          |schema_version|value  |carbon_kg|genai_tokens|
+----------------+-------------------+--------------+-------+---------+------------+
|evt_c5rhcrkjh59e|2025-08-01 22:49:00|2             |14.9331|0.002987 |NULL        |
|evt_hrqcdkf8icwj|2025-07-12 11:34:00|1             |13.1295|NULL     |NULL        |
|evt_l5ws8s604cko|2025-08-27 05:12:00|2             |104.0  |0.0208   |NULL        |
|evt_3gc4zre0xgmt|2025-08-18 23:25:00|2             |110.0  |0.022    |NULL        |
|evt_v2ntznddiaw3|2025-07-28 08:51:00|2             |14.6657|0.002933 |NULL        |
+----------------+-------------------+--------------+-------+---------+------------+
only showing top 5 rows


Empiezo con el procesamiento por streaming. Esta celda leerá hasta 10 archivos por lote, aplicará un watermark de un día y deduplicará por event_id.
Como los 120 archivos ya están cargados, availableNow los procesa y termina; el checkpoint guarda el avance de la ejecución por si pasa algo.

In [26]:
from pathlib import Path
from pyspark.sql.functions import (
    current_timestamp, to_date, input_file_name, regexp_extract
)

ruta_bronze_eventos = (
    "/content/cloud_provider/datalake/bronze/usage_events_stream"
)
ruta_checkpoint = (
    "/content/cloud_provider/datalake/checkpoints/usage_events_stream_v1"
)

eventos_stream = (
    spark.readStream
    .schema(esquema_eventos)
    .option("maxFilesPerTrigger", 10)
    .json(carpeta_eventos)
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("ingest_date", to_date("ingest_ts"))
    .withColumn(
        "source_file",
        regexp_extract(input_file_name(), r"([^/]+)$", 1)
    )
    .withColumn("event_date", to_date("timestamp"))
    .withWatermark("timestamp", "1 day")
    .dropDuplicatesWithinWatermark(["event_id"])
)

consulta = (
    eventos_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", ruta_bronze_eventos)
    .option("checkpointLocation", ruta_checkpoint)
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .start()
)

consulta.awaitTermination()

print("Streaming finalizado.")

Streaming finalizado.


**El watermark no agrega otro timestamp: establece cuánto atraso vamos a tolerar respecto del evento más reciente que Spark haya procesado.

**En el ejemplo puse 1 día. Significa que Spark tolera eventos que lleguen hasta 24 horas tarde. Si aparece uno con un timestamp anterior a ese límite, lo considera demasiado tardío para este procesamiento. No significa que descarte todo evento fechado hace más de un día respecto de hoy; los eventos son de 2025, pero Spark los compara entre sí durante la reproducción.

Luego de finalizado el streaming, hago las validaciones de id duplicados, campos nulos, etc.

In [14]:
from pyspark.sql.functions import col

eventos_bronze = spark.read.parquet(ruta_bronze_eventos)

print("Eventos escritos:", eventos_bronze.count())
print("event_id nulos:", eventos_bronze.filter(col("event_id").isNull()).count())
print("timestamp nulos:", eventos_bronze.filter(col("timestamp").isNull()).count())

print("\nEventos por versión:")
eventos_bronze.groupBy("schema_version").count().orderBy("schema_version").show()

repetidos = (
    eventos_bronze
    .groupBy("event_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
print("event_id que aparecen más de una vez:", repetidos)

print("\nEjemplos con metadatos:")
eventos_bronze.select(
    "event_id", "timestamp", "ingest_ts", "source_file", "event_date"
).show(5, truncate=False)

Eventos escritos: 7811
event_id nulos: 0
timestamp nulos: 0

Eventos por versión:
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1| 1804|
|             2| 6007|
+--------------+-----+

event_id que aparecen más de una vez: 0

Ejemplos con metadatos:
+----------------+-------------------+-----------------------+----------------------+----------+
|event_id        |timestamp          |ingest_ts              |source_file           |event_date|
+----------------+-------------------+-----------------------+----------------------+----------+
|evt_0myiduebxugo|2025-08-29 03:03:00|2026-11-18 02:18:45.873|events_part_0012.jsonl|2025-08-29|
|evt_46l2a45hgktp|2025-08-29 21:23:00|2026-11-18 02:18:45.873|events_part_0010.jsonl|2025-08-29|
|evt_gpk61ctaijqz|2025-08-29 09:32:00|2026-11-18 02:18:45.873|events_part_0018.jsonl|2025-08-29|
|evt_yu7gzy58etb3|2025-08-29 12:33:00|2026-11-18 02:18:45.873|events_part_0016.jsonl|2025-08-29|
|evt_63ji1nza4yd2|2025-08-29 18:53:

Quedaron 7.811 de los 43.200 eventos. El watermark de un día descartó muchos porque los 120 archivos ya estaban cargados y Spark los procesó fuera de orden temporal.
En otras palabras, Spark encontró archivos de fechas avanzadas antes que otros más antiguos. El margen de 1 día quedó corto y por eso no guardó todos los eventos.

Para reproducir este histórico completo, usemos un watermark de 60 días: los datos abarcan casi 60 días.

In [27]:
ruta_bronze_60d = (
    "/content/cloud_provider/datalake/bronze/usage_events_stream_wm60d"
)
ruta_checkpoint_60d = (
    "/content/cloud_provider/datalake/checkpoints/usage_events_stream_wm60d"
)

eventos_stream_60d = (
    spark.readStream
    .schema(esquema_eventos)
    .option("maxFilesPerTrigger", 10)
    .json(carpeta_eventos)
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("ingest_date", to_date("ingest_ts"))
    .withColumn(
        "source_file",
        regexp_extract(input_file_name(), r"([^/]+)$", 1)
    )
    .withColumn("event_date", to_date("timestamp"))
    .withWatermark("timestamp", "60 days")
    .dropDuplicatesWithinWatermark(["event_id"])
)

consulta_60d = (
    eventos_stream_60d.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", ruta_bronze_60d)
    .option("checkpointLocation", ruta_checkpoint_60d)
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .start()
)

consulta_60d.awaitTermination()

salida_60d = spark.read.parquet(ruta_bronze_60d)

print("Eventos leídos:", eventos_prueba.count())
print("event_id distintos en la entrada:",
      eventos_prueba.select("event_id").distinct().count())
print("Eventos escritos:", salida_60d.count())

Eventos leídos: 43200
event_id distintos en la entrada: 43200
Eventos escritos: 43200


Tardo 10 min aprox.

Los 43.200 eventos entraron y salieron, y los 43.200 event_id son distintos.

Ejecutamos reglas de calidad.

In [16]:
from pyspark.sql.functions import col, trim, coalesce

eventos = salida_60d
organizaciones = spark.read.parquet(
    "/content/cloud_provider/datalake/bronze/customers_orgs"
)

valor_presente = (
    col("value").isNotNull() &
    (trim(coalesce(col("value"), col("value"))) != "")
)

sin_unidad = eventos.filter(
    valor_presente &
    (
        col("unit").isNull() |
        (trim(coalesce(col("unit"), col("unit"))) == "")
    )
).count()

orgs_sin_match = (
    eventos.select("org_id").distinct()
    .join(organizaciones.select("org_id").distinct(), "org_id", "left_anti")
    .count()
)

print("Eventos sin event_id:", eventos.filter(col("event_id").isNull()).count())
print("Costos menores que -0.01:", eventos.filter(
    col("cost_usd_increment") < -0.01
).count())
print("Eventos con value pero sin unit:", sin_unidad)
print("org_id de eventos sin correspondencia:", orgs_sin_match)

Eventos sin event_id: 0
Costos menores que -0.01: 211
Eventos con value pero sin unit: 2038
org_id de eventos sin correspondencia: 0


Los costos negativos quedan en Silver marcados para revisión, porque podrían ser ajustes por lo que llevaran la marca "is_cost_anomally".
Los eventos que tienen un valor pero no informan su unidad van a quarantine.

In [17]:
from pyspark.sql.functions import (
    col, trim, coalesce, lit, when, concat_ws, expr
)

valor_presente = (
    col("value").isNotNull() &
    (trim(coalesce(col("value"), lit(""))) != "")
)

unidad_faltante = (
    valor_presente &
    (
        col("unit").isNull() |
        (trim(coalesce(col("unit"), lit(""))) == "")
    )
)

eventos_calidad = (
    salida_60d
    .withColumn("value_raw", col("value"))
    .withColumn("value_num", expr("try_cast(value AS DOUBLE)"))
    .withColumn(
        "is_cost_anomaly",
        col("cost_usd_increment") < -0.01
    )
    .withColumn("unit_missing_when_value_present", unidad_faltante)
    .withColumn("event_id_missing", col("event_id").isNull())
    .withColumn(
        "quality_reason",
        concat_ws(
            "|",
            when(col("event_id").isNull(), lit("EVENT_ID_NULL")),
            when(unidad_faltante, lit("UNIT_MISSING_WHEN_VALUE_PRESENT"))
        )
    )
)

eventos_quarantine = eventos_calidad.filter(
    col("event_id_missing") | col("unit_missing_when_value_present")
)

eventos_silver_candidatos = eventos_calidad.filter(
    ~col("event_id_missing") & ~col("unit_missing_when_value_present")
)

print("Candidatos a Silver:", eventos_silver_candidatos.count())
print("Registros para quarantine:", eventos_quarantine.count())
print("Costos marcados como anomalía:",
      eventos_calidad.filter(col("is_cost_anomaly")).count())

print("\nEjemplos para quarantine:")
eventos_quarantine.select(
    "event_id", "metric", "value_raw", "unit", "quality_reason"
).show(5, truncate=False)

Candidatos a Silver: 41162
Registros para quarantine: 2038
Costos marcados como anomalía: 211

Ejemplos para quarantine:
+----------------+----------------+---------+----+-------------------------------+
|event_id        |metric          |value_raw|unit|quality_reason                 |
+----------------+----------------+---------+----+-------------------------------+
|evt_hcgoqvpjpncc|storage_gb_hours|24.5432  |NULL|UNIT_MISSING_WHEN_VALUE_PRESENT|
|evt_qtc59v95o8mm|cpu_hours       |4.6971   |NULL|UNIT_MISSING_WHEN_VALUE_PRESENT|
|evt_jtk9mxjysul9|storage_gb_hours|14.3193  |NULL|UNIT_MISSING_WHEN_VALUE_PRESENT|
|evt_pmhe7h03urpj|storage_gb_hours|3.0738   |NULL|UNIT_MISSING_WHEN_VALUE_PRESENT|
|evt_vht0fb3futzt|requests        |109.0    |NULL|UNIT_MISSING_WHEN_VALUE_PRESENT|
+----------------+----------------+---------+----+-------------------------------+
only showing top 5 rows


Luego de generar una lista de eventos candidatos, avanzamos con la ingesta real a Silver.
Joineamos los eventos con la org_id de customers_orgs.

AVANZAMOS CON LA INGESTA EN SILVER.

In [18]:
from pathlib import Path
from pyspark.sql.functions import when, col

organizaciones = (
    spark.read
    .parquet("/content/cloud_provider/datalake/bronze/customers_orgs")
    .select("org_id", "org_name", "industry", "hq_region", "plan_tier")
)

silver = (
    eventos_silver_candidatos
    .join(organizaciones, on="org_id", how="left")
    .withColumn(
        "requests",
        when(col("metric") == "requests", col("value_num"))
    )
    .drop("value")
)

quarantine = eventos_quarantine.drop("value")

ruta_silver = Path(
    "/content/cloud_provider/datalake/silver/usage_events"
)
ruta_quarantine = Path(
    "/content/cloud_provider/datalake/quarantine/usage_events"
)

(
    silver.write
    .mode("overwrite")
    .partitionBy("event_date")
    .parquet(str(ruta_silver))
)

(
    quarantine.write
    .mode("overwrite")
    .partitionBy("event_date")
    .parquet(str(ruta_quarantine))
)

silver_verificado = spark.read.parquet(str(ruta_silver))
quarantine_verificado = spark.read.parquet(str(ruta_quarantine))

print("Filas en Silver:", silver_verificado.count())
print("Filas en quarantine:", quarantine_verificado.count())
print("Columnas de Silver:", silver_verificado.columns)

silver_verificado.select(
    "org_id", "org_name", "event_date", "service",
    "metric", "requests", "cost_usd_increment", "is_cost_anomaly"
).show(5, truncate=False)

Filas en Silver: 41162
Filas en quarantine: 2038
Columnas de Silver: ['org_id', 'event_id', 'timestamp', 'resource_id', 'service', 'region', 'metric', 'unit', 'cost_usd_increment', 'schema_version', 'carbon_kg', 'genai_tokens', 'ingest_ts', 'ingest_date', 'source_file', 'value_raw', 'value_num', 'is_cost_anomaly', 'unit_missing_when_value_present', 'event_id_missing', 'quality_reason', 'org_name', 'industry', 'hq_region', 'plan_tier', 'requests', 'event_date']
+------------+---------------+----------+----------+---------+--------+------------------+---------------+
|org_id      |org_name       |event_date|service   |metric   |requests|cost_usd_increment|is_cost_anomaly|
+------------+---------------+----------+----------+---------+--------+------------------+---------------+
|org_pbhsahxt|Nova Tech 1    |2025-08-23|networking|requests |107.0   |1.0701            |false          |
|org_d14ve92m|Vertex Cloud 69|2025-08-23|compute   |requests |124.0   |9.2863            |false          |


7 min aprox.

Silver y quarantine suman los 43.200 eventos.
El join con organizaciones también conservó las 41.162 filas, así que no multiplicó registros.

AVANZAMOS CON LA INGESTA EN GOLD.

Ahora agrupamos Silver por organización, día y servicio para crear el mart Gold.

In [19]:
from pyspark.sql.functions import sum as sum_, count, when, col

gold = (
    silver_verificado
    .groupBy("org_id", "org_name", "event_date", "service")
    .agg(
        sum_("cost_usd_increment").alias("daily_cost_usd"),
        sum_("requests").alias("requests"),
        sum_("genai_tokens").alias("genai_tokens"),
        sum_("carbon_kg").alias("carbon_kg"),
        count("event_id").alias("events_count"),
        sum_(
            when(col("is_cost_anomaly"), 1).otherwise(0)
        ).alias("cost_anomaly_events")
    )
    .withColumnRenamed("event_date", "usage_date")
)

ruta_gold = "/content/cloud_provider/datalake/gold/org_daily_usage_by_service"

(
    gold.write
    .mode("overwrite")
    .partitionBy("usage_date")
    .parquet(ruta_gold)
)

gold_verificado = spark.read.parquet(ruta_gold)

print("Filas del mart Gold:", gold_verificado.count())
gold_verificado.orderBy(
    "usage_date", "org_id", "service"
).show(5, truncate=False)

Filas del mart Gold: 11050
+------------+--------------+----------+-----------------+--------+------------+---------+------------+-------------------+----------+
|org_id      |org_name      |service   |daily_cost_usd   |requests|genai_tokens|carbon_kg|events_count|cost_anomaly_events|usage_date|
+------------+--------------+----------+-----------------+--------+------------+---------+------------+-------------------+----------+
|org_0lvsnujz|Nova Cloud 45 |analytics |8.873899999999999|116.0   |NULL        |NULL     |3           |0                  |2025-07-03|
|org_0lvsnujz|Nova Cloud 45 |database  |0.5723           |NULL    |NULL        |NULL     |2           |0                  |2025-07-03|
|org_0lvsnujz|Nova Cloud 45 |storage   |5.554200000000001|255.0   |NULL        |NULL     |3           |0                  |2025-07-03|
|org_0lzjjege|Alpha Cloud 77|compute   |12.6064          |120.0   |NULL        |NULL     |3           |0                  |2025-07-03|
|org_0lzjjege|Alpha Cloud 77

Valido que el total de eventos coincida con la cantidad de registros que teniamos originalmente.

In [20]:
from pyspark.sql.functions import count, sum as sum_

gold_verificado.agg(
    count("*").alias("grupos_en_gold"),
    sum_("events_count").alias("eventos_representados"),
    sum_("cost_anomaly_events").alias("anomalias_marcadas")
).show()

+--------------+---------------------+------------------+
|grupos_en_gold|eventos_representados|anomalias_marcadas|
+--------------+---------------------+------------------+
|         11050|                41162|               206|
+--------------+---------------------+------------------+



Gold quedó validado: 11.050 grupos representan los 41.162 eventos de Silver. Hay 206 anomalías de costo en Gold; las otras 5 del total de 211 quedaron en quarantine porque también tenían value sin unit.

Ejecutamos prueba de conexion con AstraDB.
Utilizamos la herramienta de Secrets para guardar el token de conexion.

In [1]:
from pathlib import Path

[p.name for p in Path("/content").glob("secure-connect-proyecto-cloud-mineria2*")]

['secure-connect-proyecto-cloud-mineria2.zip']

In [4]:
from google.colab import userdata
from pathlib import Path

token = userdata.get("ASTRA_TOKEN")
scb = list(Path("/content").glob("secure-connect-proyecto-cloud-mineria2*.zip"))

print("Token disponible:", bool(token))
print("SCB encontrado:", bool(scb))

Token disponible: True
SCB encontrado: True


Instalamos driver de Cassandra

In [5]:
%pip -q install cassandra-driver

Avanzamos con la conexion utilizando el token.

In [9]:
from google.colab import userdata
from pathlib import Path
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

token = userdata.get("ASTRA_TOKEN")
archivos_scb = list(Path("/content").glob("secure-connect-proyecto-cloud-mineria2*.zip"))

if not token:
    raise ValueError("No se encontró el secreto ASTRA_TOKEN.")
if len(archivos_scb) != 1:
    raise ValueError(f"Se esperaba encontrar un SCB y se encontraron {len(archivos_scb)}.")

cluster = Cluster(
    cloud={"secure_connect_bundle": str(archivos_scb[0])},
    auth_provider=PlainTextAuthProvider("token", token)
)
session = cluster.connect()

print("Conexión a AstraDB: OK")

Conexión a AstraDB: OK


Hacemos una prueba de conexion:

In [10]:
keyspaces = sorted({
    fila.keyspace_name
    for fila in session.execute(
        "SELECT keyspace_name FROM system_schema.keyspaces"
    )
})

print(keyspaces)

['cloud_provider', 'data_endpoint_auth', 'datastax_sla', 'system', 'system_auth', 'system_schema', 'system_traces']


In [11]:
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.org_daily_usage_by_service (
    org_id text,
    usage_month text,
    usage_date date,
    service text,
    org_name text,
    daily_cost_usd double,
    requests double,
    genai_tokens bigint,
    carbon_kg double,
    events_count int,
    cost_anomaly_events int,
    PRIMARY KEY ((org_id, usage_month), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC)
""")

print("Tabla creada o ya existente.")

Tabla creada o ya existente.


Voy a leer desde Spark el mart Gold y comprobar que vemos las 11.050 filas antes de cargarlo en AstraDB.

In [28]:
ruta_gold = "/content/cloud_provider/datalake/gold/org_daily_usage_by_service"
gold_spark = spark.read.parquet(ruta_gold)

print("Filas en Gold:", gold_spark.count())
print("Columnas:", gold_spark.columns)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/cloud_provider/datalake/gold/org_daily_usage_by_service. SQLSTATE: 42K03